# Intro to Pennylane

Partially inspired by https://pennylane.ai/demos/tutorial_qubit_rotation

In [ ]:
# pip install pennylane

Let's import pennylane!

In [ ]:
import pennylane as qp
import pennylane.numpy as np
from utils import plot_hist

Now we define the "quantum" device, namely the classical backend use to simulate, and its number of qubits, the `wires` parameter. 

For more info on the kind of backends pennylane uses look at https://docs.pennylane.ai/en/stable/code/qp_devices.html

In [ ]:
dev = qp.device("default.qubit", wires=1)

### Let's make our first circuit

Firstly, let's define the single qubit rotations we'll use:

$$R_x(\phi) = e^{-i\phi\sigma_x/2} = \begin{bmatrix}
                \cos(\phi/2) & -i\sin(\phi/2) \\
                -i\sin(\phi/2) & \cos(\phi/2)
            \end{bmatrix}.$$

$$R_y(\phi) = e^{-i\phi\sigma_y/2} = \begin{bmatrix}
                \cos(\phi/2) & -\sin(\phi/2) \\
                \sin(\phi/2) & \cos(\phi/2)
            \end{bmatrix}.$$

$$R_z(\phi) = e^{-i\phi\sigma_z/2} = \begin{bmatrix}
                e^{-i\phi/2} & 0 \\
                0 & e^{i\phi/2}
            \end{bmatrix}.$$


Let's start by implementing a simple 1-qubit circuit:
$$| \psi \rangle = R_y(\phi_2) R_x(\phi_1) | 0 \rangle.$$


![1-Qubit circuit circuit](figures/rot.png)

Use: `qp.RX(angle, wires)` and `qp.RY(angle, wires)`

In [ ]:
def circuit(params):
    # state preparation is in the zero state if not specified

    ##### TODO #####

how do we observe the outcome? Let's observe the statevector describing the state at the end of the circuit: `qp.state()`

In [ ]:
##### TODO #####

We have defined the circuit function, but we also need to add a decorator to make it runnable with the backend: `@qp.qnode(dev)`

In [ ]:
##### TODO #####

Before running the circuit, let's see how it looks

In [ ]:
params = np.array([0.54, 0.12])

qp.draw_mpl(circuit,style = "pennylane")(params)

In [ ]:
state=circuit(params)
print(state)

### Measurements

Then we measure the expectation value
$\langle \psi \mid \sigma_z \mid \psi \rangle$ of the Pauli-Z operator

$$\begin{aligned}
\sigma_z =
\begin{bmatrix} 1 &  0 \\
0 & -1
\end{bmatrix}.
\end{aligned}$$

we find that

$$\langle \psi \mid \sigma_z \mid \psi \rangle
= \langle 0 \mid R_x(\phi_1)^\dagger R_y(\phi_2)^\dagger \sigma_z  R_y(\phi_2) R_x(\phi_1) \mid 0 \rangle
= \cos(\phi_1)\cos(\phi_2).$$

the output lies between $1$ (if
$\left|\psi\right\rangle = \left|0\right\rangle$) and $-1$ (if
$\left|\psi\right\rangle = \left|1\right\rangle$).

In pennylane we need `qp.expval(qp.PauliZ(0))`


In [ ]:
##### TODO #####

In [ ]:
print(circuit(params))

We've seen how to get the statevector and how to measure the expectation value but there are other possible outcome types we can get with pennylane. 

In practice, on real devices, is not possible to get the full statevector. 

We can measure many times the output qubit according to the number of shots (`@qp.set_shots(shots=100)`) via `qp.counts()` or get the probabilities.

In [ ]:
##### TODO #####

print(f"QNode output = {circuit([0.54, 0.12])}")

In [ ]:
counts=circuit(params)
plot_hist(counts,"1 qubit circuit")

A check on the relation between amplitudes and counts:

In [ ]:
print(np.abs(state)**2, {k: v/sum(counts.values()) for k, v in counts.items()})

We can also get the probabilities with `qp.probs(wires=range(n))`. If we don't set the shots we get the ideal probabilities.

In [ ]:
##### TODO #####

print(f"QNode output = {circuit([0.54, 0.12])}")

In [ ]:
import numpy as onp
import matplotlib.pyplot as plt

dev_b = qp.device("default.qubit", wires=1)

@qp.qnode(dev_b)
def bloch(phi1, phi2):
    qp.RX(phi1, wires=0)
    qp.RY(phi2, wires=0)
    return [qp.expval(qp.PauliX(0)), qp.expval(qp.PauliY(0)), qp.expval(qp.PauliZ(0))]

phi1, phi2 = 0.54, 0.12          # try bigger angles, e.g. 1.2, 1.0, to see the arcs clearly
t = onp.linspace(0, 1, 60)
arc_x = onp.array([onp.array(bloch(phi1 * s, 0.0)) for s in t])    # RX part
arc_y = onp.array([onp.array(bloch(phi1, phi2 * s)) for s in t])   # RY part

ax = plt.figure(figsize=(6, 6)).add_subplot(projection="3d")
u, v = onp.mgrid[0:2*onp.pi:40j, 0:onp.pi:20j]
ax.plot_wireframe(onp.cos(u)*onp.sin(v), onp.sin(u)*onp.sin(v), onp.cos(v), color="lightgray", lw=0.4)
for vec, lbl in [((1.2,0,0),"x"), ((0,1.2,0),"y"), ((0,0,1.2),"|0⟩"), ((0,0,-1.2),"|1⟩")]:
    ax.plot(*zip((0,0,0), vec), color="gray", lw=0.8); ax.text(*vec, lbl)
ax.plot(*arc_x.T, lw=3, label=f"RX({phi1})")
ax.plot(*arc_y.T, lw=3, label=f"RY({phi2})")
ax.quiver(0, 0, 0, *arc_y[-1], color="k", lw=2, arrow_length_ratio=0.1)
ax.set_box_aspect((1, 1, 1)); ax.set_axis_off(); ax.legend()
print("Final Bloch vector (x, y, z):", arc_y[-1].round(4), " | <Z> =", round(arc_y[-1][2], 4))
plt.show()

## More Qubits!

Now let's introduce two-qubit gates with the following Bell state preparation circuit: we need a Hadamard gate on the first qubit and a CNOT acting on the two qubits.

In [ ]:
dev = qp.device('default.qubit', wires=2)

@qp.set_shots(shots=100)
@qp.qnode(dev)
def circuit():
    ##### TODO #####
    return qp.probs(wires=range(2))

In [ ]:
qp.draw_mpl(circuit,style = "pennylane")()
print(f"QNode output = {circuit()}")

Let's just get the counts with `qp.counts()` and visualize the outcomes:

In [ ]:
##### TODO #####

In [ ]:
counts=circuit()
plot_hist(counts,"Bell State")

# Let's run our first quantum algorithm

Next notebook!

